In [ ]:
import sys
from pathlib import Path

# repo root, whether the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from math import ceil

from src.data import load_sow, load_sow_counterpart, load_customers
from src.features import build_customer_features

sow = load_sow()
sow_cp = load_sow_counterpart()

# Keep only real merchants and the "other" bucket (drops payment and financial providers)
merchants = sow_cp[sow_cp['counterpart_type'].isin(['merchant', 'other'])].copy()

# Spend per category and counterpart, plus share of category spend
top_all = merchants.groupby(['category', 'top_counterpart'])['total_amount'].sum().reset_index()
category_totals = (merchants.groupby('category')['total_amount'].sum()
                   .rename('category_total').reset_index())
top_all = top_all.merge(category_totals, on='category')
top_all['share'] = top_all['total_amount'] / top_all['category_total']

# Top 10 per category
top10_sorted = (top_all.sort_values(['category', 'share'], ascending=[True, False])
                       .groupby('category').head(10))

In [ ]:
# SoW per counterpart within each category over the whole period
# Filter out payment providers AND financial providers
# Keep only merchants

merchant_types = ['merchant', 'other']
merchants = sow_cp[sow_cp['counterpart_type'].isin(merchant_types)].copy()

print(f"Rows before: {len(sow_cp)}, after: {len(merchants)}")
print(f"\nCounterpart types remaining: {merchants['counterpart_type'].unique()}")

In [ ]:
# Aggregate by category & top_counterpart
top_all = merchants.groupby(['category', 'top_counterpart'])['total_amount'].sum().reset_index()

# Total per category
category_totals = merchants.groupby('category')['total_amount'].sum().reset_index()
category_totals.columns = ['category', 'category_total']

# Merge & calculate share
top_all = top_all.merge(category_totals, on='category')
top_all['share'] = top_all['total_amount'] / top_all['category_total']

# Top 10 per category
top10 = top_all.sort_values(['category', 'share'], ascending=[True, False]).groupby('category').head(10)

print(f"Top 10 shape: {top10.shape}")
print("\nTop 10 Preview:")
print(top10.head(15))

In [ ]:
import plotly.graph_objects as go
import plotly.subplots as sp
from math import ceil

# Identify categories with only "other"
cats_with_merchants = []
cats_only_other = []

for cat in sorted(top10_sorted['category'].unique()):
    cat_data = top10_sorted[top10_sorted['category'] == cat]
    if len(cat_data[cat_data['top_counterpart'] != 'other']) > 0:
        cats_with_merchants.append(cat)
    else:
        cats_only_other.append(cat)

# Avg. monthly customers per counterpart, shown next to the share
avg_cust = (merchants.groupby(['category', 'top_counterpart'])['n_customers']
            .mean().rename('avg_cust').reset_index())
top10_sorted = (top10_sorted.drop(columns='avg_cust', errors='ignore')
                .merge(avg_cust, on=['category', 'top_counterpart'], how='left'))

# Reorder: merchants first, then unmapped-only
categories = cats_with_merchants + cats_only_other

n_cats = len(categories)
cols = 3
rows = ceil(n_cats / cols)

fig = sp.make_subplots(
    rows=rows, cols=cols,
    subplot_titles=categories,
    specs=[[{'type': 'bar'}] * cols for _ in range(rows)],
    vertical_spacing=0.08,  # ← HERE, not in update_layout
    horizontal_spacing=0.08  # ← HERE
)

for idx, cat in enumerate(categories):
    row = (idx // cols) + 1
    col = (idx % cols) + 1
    
    cat_data = top10_sorted[top10_sorted['category'] == cat].copy()
    
    # Sort: non-"other" first, then "other" at END
    cat_data = pd.concat([
        cat_data[cat_data['top_counterpart'] != 'other'].sort_values('share', ascending=True),
        cat_data[cat_data['top_counterpart'] == 'other'].sort_values('share', ascending=True)
    ])
    
    # Grey for "other", steelblue for merchants
    colors = ['lightgrey' if x == 'other' else 'steelblue' for x in cat_data['top_counterpart']]
    
    fig.add_trace(
        go.Bar(
            y=cat_data['top_counterpart'], 
            x=cat_data['share'], 
            orientation='h', 
            showlegend=False,
            marker_color=colors,
            text=[f"{s:.0%} · {c:,.0f} cust./month" for s, c in zip(cat_data['share'], cat_data['avg_cust'])],
            textposition='auto',
            textfont=dict(size=8)
        ),
        row=row, col=col
    )

# UPDATE LAYOUT (without spacing)
fig.update_layout(
    height=2400,
    width=1200,
    showlegend=False, 
    title_text="Top 10 Merchants per Category (Ordered by Data Richness)",
    margin=dict(l=80, r=20, t=60, b=40)
)

fig.update_xaxes(title_text="Share", title_font=dict(size=10), tickfont=dict(size=8))
fig.update_yaxes(tickfont=dict(size=9))

fig.show()

In [ ]:
import numpy as np

# Avg. monthly customers per counterpart (averaging avoids double-counting across months)
cust = (merchants.groupby(['category', 'top_counterpart'])['n_customers']
        .mean().rename('avg_monthly_customers'))
top_all = (top_all.drop(columns='avg_monthly_customers', errors='ignore')
                  .join(cust, on=['category', 'top_counterpart']))

rows = []
for cat, g in top_all.groupby('category'):
    other_share = g.loc[g['top_counterpart'] == 'other', 'share'].sum()
    named = g[g['top_counterpart'] != 'other'].sort_values('share', ascending=False)

    rows.append({
        'category': cat,
        'total_amount': g['total_amount'].sum(),
        'other_share': other_share,
        'n_named': len(named),
        # share of TOTAL category spend held by the 3 largest named merchants
        'top3_share': named['share'].head(3).sum(),
        # HHI among named merchants only (rescaled to sum to 1), NaN if none
        'hhi_named': ((named['share'] / named['share'].sum()) ** 2).sum() if len(named) else np.nan,
    })

concentration_df = pd.DataFrame(rows).sort_values('top3_share', ascending=False).reset_index(drop=True)

# Sanity checks that can actually fail
for _, r in concentration_df.iterrows():
    named_total = top_all[(top_all['category'] == r['category']) & (top_all['top_counterpart'] != 'other')]['share'].sum()
    assert abs(named_total + r['other_share'] - 1) < 1e-9, f"shares != 1 in {r['category']}"
    assert r['top3_share'] <= 1 - r['other_share'] + 1e-9, f"top3 > named total in {r['category']}"

display(concentration_df.style.format({
    'total_amount': '{:,.0f}', 'other_share': '{:.1%}', 'top3_share': '{:.1%}', 'hhi_named': '{:.3f}'
}))

In [ ]:
from src import mappings as M

def quarterly_shares(category, groups, order):
    """Share of category spend per quarter by counterpart group (in %)."""
    d = merchants[merchants['category'] == category].copy()
    d['grp'] = d['top_counterpart'].map(groups).fillna('other')
    p = d.pivot_table(index='quarter', columns='grp', values='total_amount',
                      aggfunc='sum', fill_value=0)
    p = p.reindex(columns=order, fill_value=0)
    return p.div(p.sum(axis=1), axis=0) * 100

def stacked_area(p, title):
    fig = go.Figure()
    x = p.index.to_timestamp()
    for col in p.columns:
        fig.add_trace(go.Scatter(
            x=x, y=p[col], name=col, stackgroup='one', mode='lines',
            line=dict(width=0.5),
            fillcolor='lightgrey' if col == 'other' else None,
            line_color='grey' if col == 'other' else None))
    fig.update_layout(title=title, yaxis_title='% of category spend',
                      height=400, width=900, legend_title_text='')
    return fig

# Groceries: max 4 named series + other
groc_groups = {'coop': 'coop', 'migros': 'migros',
               'lidl': 'discounter', 'aldi': 'discounter', 'denner': 'discounter'}
groc = quarterly_shares('groceries', groc_groups, ['coop', 'migros', 'discounter', 'other'])
stacked_area(groc, 'Groceries: share of category spend per quarter').show()

# Transport: parking is folded into other (max 4 named series)
trans_groups = {k: v for k, v in M.TRANSPORT_GROUPS.items() if v != 'parking'}
trans = quarterly_shares('transport', trans_groups,
                         ['fuel', 'public transport', 'ride hailing', 'micromobility', 'other'])
stacked_area(trans, 'Transport: share of category spend per quarter').show()

In [ ]:
for cat in ['communication', 'shopping']:
    top = (merchants[(merchants['category'] == cat) & (merchants['top_counterpart'] != 'other')]
           .groupby('top_counterpart')['total_amount'].sum().sort_values(ascending=False).head(8))
    print(cat, '->', list(top.index))

In [ ]:
# Communication: 4 named series (Apple, Google, Post, Spotify), rest is "other"
comm_groups = {'apple': 'apple', 'google': 'google', 'post': 'post', 'spotify': 'spotify'}
comm = quarterly_shares('communication', comm_groups, ['apple', 'google', 'post', 'spotify', 'other'])
stacked_area(comm, 'Communication: share of category spend per quarter').show()

# Shopping: 4 named series, rest is "other"
shop_groups = {'digitec galaxus': 'digitec galaxus', 'amazon': 'amazon',
               'zalando': 'zalando', 'ikea': 'ikea'}
shop = quarterly_shares('shopping', shop_groups, ['digitec galaxus', 'amazon', 'zalando', 'ikea', 'other'])
stacked_area(shop, 'Shopping: share of category spend per quarter').show()

In [ ]:
# Gambling: like-for-like series (jackpots.ch removed from numerator AND denominator)
core = ['interwetten', 'mycasino.ch', 'swiss casinos', 'swisslos']
e = merchants[(merchants['category'] == 'entertainment') &
              (merchants['top_counterpart'] != 'jackpots.ch')].copy()
e['gam'] = e['top_counterpart'].isin(core)
q = e.groupby(['quarter', 'gam'])['total_amount'].sum().unstack(fill_value=0)
q.columns = ['rest_of_entertainment', 'gambling_core']   # columns are ordered False, True

# share for the chart
core_share = (q['gambling_core'] / q.sum(axis=1) * 100).round(1)

# index check (Q1 2021 = 100): does gambling grow slower than the rest?
(q / q.iloc[0] * 100).round(0)

In [ ]:
# Gambling chart: like-for-like series vs. series incl. jackpots.ch (break in Q2 2022)
core = ['interwetten', 'mycasino.ch', 'swiss casinos', 'swisslos']
all_share = quarterly_shares('entertainment', M.ENTERTAINMENT_GROUPS,
                             ['gambling', 'streaming', 'gaming', 'other'])['gambling']
x = [str(p) for p in core_share.index]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=all_share.values, name='incl. jackpots.ch (series break)',
                         mode='lines+markers', line=dict(color='lightgrey', dash='dot')))
fig.add_trace(go.Scatter(x=x, y=core_share.values, name='like-for-like (excl. jackpots.ch)',
                         mode='lines+markers'))
fig.add_shape(type='line', x0=4.5, x1=4.5, y0=0, y1=1, xref='x', yref='paper',
              line=dict(color='grey', width=1, dash='dash'))
fig.add_annotation(x=4.5, y=1, yref='paper', text='jackpots.ch disappears',
                   showarrow=False, yshift=0)
fig.update_layout(title='Gambling: share of entertainment merchant spend per quarter',
                  yaxis_title='% of entertainment spend', height=400, width=900,
                  legend=dict(orientation='h', y=-0.2), margin=dict(t=80))
fig.show()

In [ ]:
from src.data import monthly_active

# Seasonality: spend per active customer, indexed to its own calendar-year mean
active = monthly_active(sow)
cats = (sow.groupby('category')['total_amount'].sum()
           .sort_values(ascending=False).head(9).index.tolist())

m = (sow[sow['category'].isin(cats)]
     .pivot_table(index='date', columns='category', values='total_amount', aggfunc='sum')
     .reindex(columns=cats))
per_cust = m.div(active, axis=0)
idx = per_cust / per_cust.groupby(per_cust.index.year).transform('mean')
season = idx.groupby(idx.index.month).mean()   # rows: month 1..12

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig = sp.make_subplots(rows=3, cols=3, subplot_titles=cats,
                       shared_yaxes=True, vertical_spacing=0.12, horizontal_spacing=0.05)
for i, cat in enumerate(cats):
    fig.add_trace(go.Scatter(x=months, y=season[cat], mode='lines+markers', showlegend=False),
                  row=i // 3 + 1, col=i % 3 + 1)
    fig.add_hline(y=1, line=dict(color='lightgrey', dash='dash'), row=i // 3 + 1, col=i % 3 + 1)
fig.update_layout(title='Seasonality: month-of-year index (1.0 = average month)',
                  height=800, width=1000)
fig.update_xaxes(tickfont=dict(size=8))
fig.show()

# Strongest peak and trough per category
pd.DataFrame({'peak_month': season.idxmax(), 'peak': season.max().round(2),
              'trough_month': season.idxmin(), 'trough': season.min().round(2)})

### Robustness: seasonality without 2021

Restaurants and holidays are distorted by Covid restrictions in H1 2021
(e.g. the March trough of holidays disappears when 2021 is excluded).
Peak months are stable across both versions; trough months of flat categories
are not, so only the 2022-2023 ranges of the large swings are reported.

In [ ]:
idx_late = idx[idx.index.year >= 2022]
season_late = idx_late.groupby(idx_late.index.month).mean()

pd.DataFrame({
    'peak_all': season.max().round(2), 'peak_22_23': season_late.max().round(2),
    'trough_all': season.min().round(2), 'trough_22_23': season_late.min().round(2),
    'peak_month_22_23': season_late.idxmax(), 'trough_month_22_23': season_late.idxmin(),
})

In [ ]:
# Category share of the total card spend per year (all categories, from sow)
y = sow.assign(year=sow['date'].dt.year)
tot = y.pivot_table(index='category', columns='year', values='total_amount', aggfunc='sum', fill_value=0)
share_y = tot.div(tot.sum(), axis=1) * 100
share_y['change_pp'] = share_y[2023] - share_y[2021]
share_y.sort_values('change_pp').round(1)

In [ ]:
# Cash + savings combined, per quarter (share of total card spend)
qt = sow.pivot_table(index='quarter', columns='category', values='total_amount', aggfunc='sum', fill_value=0)
tot_q = qt.sum(axis=1)
leak = pd.DataFrame({
    'cash': qt['cash'] / tot_q * 100,
    'savings': qt['savings'] / tot_q * 100,
})
leak['cash+savings'] = leak['cash'] + leak['savings']
leak.round(1)

In [ ]:
from src.data import monthly_active

tot_spend = sow.groupby('category')['total_amount'].sum()
cust_share = sow.groupby('category')['n_customers'].mean() / monthly_active(sow).mean() * 100
pos = (sow['pos_perc'] * sow['total_amount']).groupby(sow['category']).sum() / tot_spend * 100
named = top_all[top_all['top_counterpart'] != 'other']
top_cp = (named.sort_values('total_amount', ascending=False)
               .groupby('category').head(1).set_index('category')['top_counterpart'])

cat_table = pd.DataFrame({
    'total_spend_mchf': tot_spend / 1e6,
    'share_of_total_pct': tot_spend / tot_spend.sum() * 100,
    'active_customers_pct': cust_share,
    'top_counterpart': top_cp,
    'top3_share_pct': concentration_df.set_index('category')['top3_share'] * 100,
    'pos_pct': pos,
    'ecom_pct': 100 - pos,
    'trend_pp_2021_23': share_y['change_pp'],
}).sort_values('share_of_total_pct', ascending=False)

# cash / savings: all named counterparts are financial providers, so no merchant is listed
# No named merchants -> concentration is undefined, not 0
cat_table['top3_share_pct'] = cat_table['top3_share_pct'].where(cat_table['top3_share_pct'] > 0)
# Groceries is the reference for "active customers" (max over categories), so 100 % by construction
cat_table.loc['groceries', 'active_customers_pct'] = np.nan

cat_table.round(1)

## Future: forecast of the monthly SoW share

Four monthly series, 36 points each (Jan 2021 – Dec 2023):

- share of Coop, Migros and discounters (Lidl, Aldi, Denner) in grocery merchant spend (same base as the stacked area above)
- Revolut share of total card spend (cash + savings rows, feeds issue 06)

Models are kept simple because of the short history: seasonal naive (same month last year), ETS with damped additive trend and additive seasonality (`ETSModel`, the state-space version of `ExponentialSmoothing`, which gives prediction intervals) and SARIMA(1,0,0)(0,1,0)<sub>12</sub>. Backtest: fit on the first 30 months, forecast the last 6, MAE in percentage points. The final forecast refits the model with the lowest backtest MAE on all 36 months and shows the 80 % band. With only 6 test points the MAE ranking is noisy, the differences between models are within a few tenths of a point.

In [ ]:
import warnings
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Monthly shares in %
g = merchants[merchants['category'] == 'groceries'].copy()
g['grp'] = g['top_counterpart'].map(M.GROCERY_GROUPS).fillna('other')
gp = g.pivot_table(index='date', columns='grp', values='total_amount', aggfunc='sum', fill_value=0)
groc_m = gp.div(gp.sum(axis=1), axis=0) * 100

total_m = sow.groupby('date')['total_amount'].sum()
revolut_m = (sow_cp[sow_cp['top_counterpart'] == 'revolut']
             .groupby('date')['total_amount'].sum().reindex(total_m.index, fill_value=0))

series = {
    'Coop (% of groceries)': groc_m['coop'],
    'Migros (% of groceries)': groc_m['migros'],
    'Discounters (% of groceries)': groc_m['discounter'],
    'Revolut (% of card spend)': revolut_m / total_m * 100,
}
series = {k: v.asfreq('MS') for k, v in series.items()}
assert all(len(s) == 36 and s.notna().all() for s in series.values())
pd.DataFrame(series).describe().round(2)

In [ ]:
H, TEST, ALPHA = 12, 6, 0.2   # forecast horizon, backtest months, 80 % band

def seasonal_naive(y, h):
    mean = np.resize(y.iloc[-12:].values, h)
    # band from the spread of the year-on-year differences
    sd = (y - y.shift(12)).dropna().std()
    return mean, np.c_[mean - 1.2816 * sd, mean + 1.2816 * sd]

def ets(y, h):
    r = ETSModel(y, error='add', trend='add', damped_trend=True,
                 seasonal='add', seasonal_periods=12).fit(disp=False)
    f = r.get_prediction(start=len(y), end=len(y) + h - 1).summary_frame(alpha=ALPHA)
    return f['mean'].values, f[['pi_lower', 'pi_upper']].values

def sarima(y, h):
    r = SARIMAX(y, order=(1, 0, 0), seasonal_order=(0, 1, 0, 12)).fit(disp=False)
    f = r.get_forecast(h)
    return f.predicted_mean.values, f.conf_int(alpha=ALPHA).values

MODELS = {'seasonal naive': seasonal_naive, 'ETS (A,Ad,A)': ets, 'SARIMA(1,0,0)(0,1,0)12': sarima}

backtest, bt_pred = {}, {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for name, y in series.items():
        train, test = y.iloc[:-TEST], y.iloc[-TEST:]
        backtest[name] = {}
        for m, fn in MODELS.items():
            pred, _ = fn(train, TEST)
            backtest[name][m] = np.abs(pred - test.values).mean()
            bt_pred[(name, m)] = pd.Series(pred, index=test.index)

mae = pd.DataFrame(backtest).T
mae['best'] = mae.idxmin(axis=1)
mae.round(2)

In [ ]:
# Refit the best model per series on all 36 months and forecast 12 months
fc = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for name, y in series.items():
        best = mae.loc[name, 'best']
        mean, band = MODELS[best](y, H)
        idx = pd.date_range(y.index[-1] + pd.offsets.MonthBegin(), periods=H, freq='MS')
        fc[name] = pd.DataFrame({'mean': mean, 'lo': band[:, 0], 'hi': band[:, 1]}, index=idx)

titles = [f"{n} · {mae.loc[n, 'best']}, MAE {mae.loc[n, mae.loc[n, 'best']]:.2f} pp" for n in series]
fig = sp.make_subplots(rows=2, cols=2, subplot_titles=titles, vertical_spacing=0.14, horizontal_spacing=0.08)
for i, (name, y) in enumerate(series.items()):
    r, c = i // 2 + 1, i % 2 + 1
    f, best = fc[name], mae.loc[name, 'best']
    # backtest window shaded
    ax = '' if i == 0 else i + 1
    fig.add_shape(type='rect', x0=y.index[-TEST] - pd.DateOffset(days=15), x1=y.index[-1] + pd.DateOffset(days=15),
                  y0=0, y1=1, xref=f'x{ax}', yref=f'y{ax} domain',
                  fillcolor='lightgrey', opacity=0.35, line_width=0, layer='below')
    fig.add_trace(go.Scatter(x=y.index, y=y, mode='lines', line=dict(color='#2a78d6'),
                             name='actual', showlegend=i == 0), row=r, col=c)
    fig.add_trace(go.Scatter(x=bt_pred[(name, best)].index, y=bt_pred[(name, best)], mode='lines',
                             line=dict(color='#eb6834', dash='dot'), name='backtest forecast',
                             showlegend=i == 0), row=r, col=c)
    fig.add_trace(go.Scatter(x=list(f.index) + list(f.index[::-1]), y=list(f['hi']) + list(f['lo'][::-1]),
                             fill='toself', fillcolor='rgba(235,104,52,0.18)', line=dict(width=0),
                             name='80 % band', showlegend=i == 0, hoverinfo='skip'), row=r, col=c)
    fig.add_trace(go.Scatter(x=f.index, y=f['mean'], mode='lines', line=dict(color='#eb6834'),
                             name='forecast 2024', showlegend=i == 0), row=r, col=c)
fig.update_yaxes(title_text='%', title_font=dict(size=10))
fig.update_annotations(font_size=11)
fig.update_layout(title='SoW forecast 12 months ahead (grey: 6-month backtest window)',
                  height=700, width=1100, legend=dict(orientation='h', y=-0.08))
fig.show()

# Forecast summary: 2023 actual mean vs 2024 forecast mean
pd.DataFrame({
    'mean_2023': {n: y['2023'].mean() for n, y in series.items()},
    'forecast_2024': {n: fc[n]['mean'].mean() for n in series},
    'band_2024_lo': {n: fc[n]['lo'].mean() for n in series},
    'band_2024_hi': {n: fc[n]['hi'].mean() for n in series},
}).round(1)

### What influences it

The external series (LIK food index, fuel prices, events calendar) come from issue 11 and are not in the repo yet. Until then:

- **Covid 2021H1**: restaurants and holidays are depressed in H1 2021 (see the seasonality robustness check), which inflates their 2021 → 2023 gain in the category comparison.
- **Calendar**: December peaks in shopping, groceries and cash, July peaks in restaurants, holidays and transport.
- **Prices**: the discounter share is a ratio inside groceries, so a food price index that hits all retailers alike cancels out. Deflating only matters for spend per customer (real vs nominal). This needs the LIK series from issue 11, then: `real = nominal / (LIK_food / LIK_food[2021-01])`.
- **Revolut**: Revolut moves from `savings` to `cash` in 2022Q1 (recategorisation in the source, see A1). The share is taken over both categories, so the move itself does not create a jump.

## Predict SoW per customer (optional)

Lukas' "estimate how high / low the SoW". Only possible at category level: how well do behaviour features explain a customer's grocery share of total spend? Category shares are left out of the features (they add up to 1 with the target), customers with fewer than 10 transactions are dropped (their shares are noise).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.inspection import permutation_importance
from src.data import load_customer_table

feats = build_customer_features(load_customer_table())
feats = feats[feats['n_transactions'] >= 10]
X_cols = ['log_n_transactions', 'log_total_amount', 'log_n_counterparts', 'log_days_active', 'log_n_country',
          'tx_per_day', 'amt_per_tx', 'cp_per_tx', 'pos_perc', 'n_categories', 'n_currencies',
          'foreign_tx_share', 'chf_share']
X, y_g = feats[X_cols], feats['cat_groceries_share']

gbr = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, random_state=0)
r2 = cross_val_score(gbr, X, y_g, cv=KFold(5, shuffle=True, random_state=0), scoring='r2')
print(f"n = {len(X):,}, grocery share mean {y_g.mean():.1%}, CV R² = {r2.mean():.2f} ± {r2.std():.2f}")

imp = permutation_importance(gbr.fit(X, y_g), X, y_g, n_repeats=5, random_state=0)
pd.Series(imp.importances_mean, index=X_cols).sort_values(ascending=False).head(6).round(3)

In [ ]:
# Direction of the strongest drivers: mean grocery share (%) by quintile
out = {}
for c in ['amt_per_tx', 'pos_perc', 'cp_per_tx', 'foreign_tx_share']:
    q = pd.qcut(feats[c].rank(method='first'), 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
    out[c] = feats.groupby(q, observed=True)['cat_groceries_share'].mean() * 100
pd.DataFrame(out).T.round(1)